
```markdown
This analysis is to come up with a trading indicator driven by volume surge. Other secondary indicators can be added to optimize the performance.

## Features of This Model
- Adaptive Entry/Exit Signals using VMA-ROC with rolling standard deviation
- Risk Management with Stop-Loss (2%) & Trailing Stop (1.5%) (Needs further fine-tuning)
- Improved Signal Accuracy by using longer lookback periods & moving averages (Needs further fine-tuning)
- Backtested on Real Stock Data to validate performance
```

In [ ]:
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt

import os
import sys

sys.path.append(os.path.dirname(os.path.realpath('__file__')) + '/../../')
from src.common.constants import *

This function is to download historical data to specified directory

In [ ]:
from MomentumLearning import Utils
import pandas as pd

# granularities = ["1m", "2m", "5m", "15m", "30m", "60m", "1d" ]
granularities = ["2m"]
# start_date_to_download = '2025-01-18'
# end_date_to_download = '2025-01-21'

if __name__ == "__main__":   
    dirname = GlobalConstants.historicMarketData_dir
    symbolListDataframe = pd.read_csv(GlobalConstants.historicMarketData_dir + 'equitySchema.csv')
    symbolList = Utils.panda_series_toList_converter(symbolListDataframe['Symbol'])  
    
    print(symbolList) 
    
    for granularity in granularities: 
        Utils.download_market_data(symbolList, granularity, dirname, ".csv")

In [ ]:
# file_pattern = GlobalConstants.historicMarketData_dir + '2m/RELIANCE.NS_2025*.csv'
file_pattern = GlobalConstants.historicMarketData_dir + '2m/INFY.NS*.csv'

# Read all CSV files in the current directory
csv_files = glob.glob(file_pattern)

# Create an empty list to store dataframes
dataframes = []

# Loop through the list of CSV files and read each one into a dataframe
for file in csv_files:
    df = pd.read_csv(file)
    dataframes.append(df)

# Merge all dataframes into a single dataframe
merged_df = pd.concat(dataframes, ignore_index=True)

# Convert the 'Datetime' column to datetime format
merged_df['Datetime'] = pd.to_datetime(merged_df['Datetime'])

# Sort the dataframe by the 'Datetime' column
merged_df = merged_df.sort_values(by='Datetime').reset_index(drop=True)

merged_df.set_index('Datetime', inplace=True)

# merged_df['Close'].plot()

df = merged_df.copy()

In [ ]:
# Optimized Parameters
vma_window = 20  # Moving Average of Volume
roc_lookback = 14  # Lookback for VMA-ROC
rolling_window = 100  # Window for calculating standard deviation
min_threshold = 20  # Minimum threshold to avoid too many signals
stop_loss_pct = 2  # 2% Stop Loss
trailing_stop_pct = 1.5  # 1.5% Trailing Stop Loss
candle_lookback = 5  # Lookback for VMA-ROC Uptrend
atr_window = 14  # ATR period

# Compute VMA (Volume Moving Average)
df['VMA'] = df['Volume'].rolling(window=vma_window).mean()

# Compute VMA-ROC (Rate of Change of Volume MA)
df['VMA_ROC'] = ((df['VMA'] - df['VMA'].shift(roc_lookback)) / df['VMA'].shift(roc_lookback)) * 100

# Compute Rolling Standard Deviation of VMA-ROC
df['VMA_ROC_STD'] = df['VMA_ROC'].rolling(window=rolling_window).std()

# Define Dynamic Threshold: 1.5 * Standard Deviation
df['Dynamic_Threshold'] = 1.5 * df['VMA_ROC_STD']

# Set a Minimum Threshold to Prevent Overfitting in Quiet Markets
df['Final_Threshold'] = df[['Dynamic_Threshold']].applymap(lambda x: max(x, min_threshold))

# Ensure a Buy Signal is generated only if VMA-ROC is in an uptrend over the past 5 candles
df['VMA_ROC_Uptrend'] = df['VMA_ROC'].rolling(window=candle_lookback).apply(lambda x: all(x[i] > x[i-1] for i in range(1, len(x))), raw=True)

# ATR-Based Stop-Loss & Trailing Stop
df['ATR'] = df['High'].rolling(window=atr_window).max() - df['Low'].rolling(window=atr_window).min()

# Define dynamic stop-loss and trailing stop based on ATR
df['Stop_Loss'] = df['Close'] - (1.5 * df['ATR'])
df['Trailing_Stop'] = df['Close'] - (1.5 * df['ATR'])

# Filter Trades by Market Hours (Avoid Low-Liquidity Periods)
# df_filtered = pd.concat([df.between_time('09:31', '13:30'), df.between_time('13:31', '15:29')])

df_filtered = df

# Trend Confirmation Using 50-EMA
df_filtered['EMA_50'] = df_filtered['Close'].ewm(span=50, adjust=False).mean()

df_filtered

# print all the rows for 2024-12-19
# print(df_filtered.loc['2024-12-19'])


In [ ]:
# Update Buy Signal: VMA-ROC should be above Final_Threshold **and** both volume and price be in an uptrend
df_filtered['Buy_Signal'] = (df_filtered['VMA_ROC'] > df_filtered['Final_Threshold']) & (df_filtered['VMA_ROC_Uptrend'] == 1) & (df_filtered['Close'] > df_filtered['EMA_50'])

# Sell Signal: When VMA-ROC crosses below -dynamic threshold
df_filtered['Sell_Signal'] = (df_filtered['VMA_ROC'] < -df_filtered['Final_Threshold']) & (df_filtered['Close'] < df_filtered['EMA_50'])

# # print dataframe with buy signal as True
# df[df['Buy_Signal'] == True]

# filter dataframe for 2025-01-17
# df = df.loc['2025-01-17']

# print(df_filtered.loc['2024-12-19'])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(df_filtered.index, df_filtered['VMA_ROC'], label="VMA-ROC", color='blue')
plt.plot(df_filtered.index, df_filtered['Final_Threshold'], label="Buy Threshold", linestyle="dashed", color='green')
plt.plot(df_filtered.index, -df_filtered['Final_Threshold'], label="Sell Threshold", linestyle="dashed", color='red')

# Mark Buy and Sell Signals
plt.scatter(df_filtered.index[df_filtered['Buy_Signal']], df_filtered['VMA_ROC'][df_filtered['Buy_Signal']], marker="^", color="green", label="Buy Signal", alpha=1)
plt.scatter(df_filtered.index[df_filtered['Sell_Signal']], df_filtered['VMA_ROC'][df_filtered['Sell_Signal']], marker="v", color="red", label="Sell Signal", alpha=1)

plt.title("VMA-ROC with Adaptive Threshold on Reliance (2-min Candles, Jan)")
plt.xlabel("Time")
plt.ylabel("VMA-ROC (%)")
plt.legend()
plt.show()

In [ ]:
# Backtesting with Updated Strategy
initial_capital = 100000
capital = initial_capital
position = 0
entry_price = 0
highest_price = 0
trade_log = []
returns = []

# print(df_filtered.loc['2024-12-19'])
# df_filtered = df_filtered.loc['2024-12-19']

for i in range(len(df_filtered)):
    row = df_filtered.iloc[i]
    
    # print(row.name, row['Close'], row['Buy_Signal'], row['Sell_Signal'])

    # if position > 0 and row.name.time() == pd.to_datetime('13:27').time():
    #     print('Holding Position:', position, 'at', row['Close'])       
        
    
    # Execute Buy Signal (Enter a long position)
    if row['Buy_Signal'] and position == 0 and row.name.time() <= pd.to_datetime('15:15').time()\
            and row.name.time() >= pd.to_datetime('09:45').time():
        position = capital // row['Close']
        entry_price = row['Close']
        highest_price = entry_price
        capital -= position * entry_price
        trade_log.append(('BUY', row.name, entry_price, position, capital, 0))

    # Track Trailing Stop
    if position > 0:
        highest_price = max(highest_price, row['Close'])
        trailing_stop_price = highest_price - (1.5 * row['ATR'])

    # Execute Sell Signal, Stop Loss, or Trailing Stop or time is past 3:25 PM as Final Exit
    # This is because strategy doesn't want to keep overnight risk
    if position > 0:
        stop_loss_price = entry_price - (1.5 * row['ATR'])

        if row['Sell_Signal'] or row['Close'] <= stop_loss_price or row['Close'] <= trailing_stop_price or row.name.time() >= pd.to_datetime('15:25').time():
            sell_price = row['Close']
            capital += position * sell_price
            trade_return = (sell_price - entry_price) / entry_price
            returns.append(trade_return)
            trade_log.append(('SELL', row.name, sell_price, position, capital, trade_return*100))
            position = 0

# Final Exit if Position Remains Open
if position > 0:
    sell_price = df_filtered['Close'].iloc[-1]
    capital += position * sell_price
    trade_return = (sell_price - entry_price) / entry_price
    returns.append(trade_return)
    trade_log.append(('FINAL EXIT', df_filtered.index[-1], sell_price, position, capital, trade_return*100))
    position = 0



In [ ]:
# Calculate Performance Metrics
net_profit = capital - initial_capital
roi = (net_profit / initial_capital) * 100

# Convert trade log to DataFrame
trade_df = pd.DataFrame(trade_log, columns=['Action', 'Datetime', 'Price', 'Shares', 'Capital', 'Return%'])

# Calculate Additional Performance Metrics
hpr = np.mean(returns) if returns else np.nan # Holding Period Return
twr = np.prod([1 + r for r in returns]) - 1 if returns else np.nan # Time-Weighted Return

# Risk Metrics
returns_series = pd.Series(returns) if returns else pd.Series([0])
sharpe_ratio = returns_series.mean() / returns_series.std() * np.sqrt(252) if returns_series.std() > 0 else np.nan  # Assume 252 trading days

# Maximum Drawdown
cumulative_returns = (1 + returns_series).cumprod() if returns else pd.Series([1])
peak = cumulative_returns.cummax()
drawdown = (cumulative_returns - peak) / peak
max_drawdown = drawdown.min() if not drawdown.empty else np.nan

# Annualized Return
trading_days = len(df_filtered) / 195  # Assume 195 trading days in a year
annualized_return = ((1 + twr) ** (252 / trading_days)) - 1 if not np.isnan(twr) else np.nan

# Annualized Volatility
annualized_volatility = returns_series.std() * np.sqrt(252) if returns_series.std() > 0 else np.nan

# Value at Risk (VaR) at 95% confidence level (assuming normal distribution)
var_95 = np.percentile(returns_series, 5) if not returns_series.empty else np.nan

# Win Rate (Percentage of profitable trades)
win_rate = sum(1 for r in returns if r > 0) / len(returns) if returns else 0

# Store Updated Performance Metrics
performance_metrics = {
    "Net Profit (₹)": round(net_profit, 2),
    "ROI (%)": round(roi, 2),
    "Holding Period Return (HPR)": round(hpr, 4) if not np.isnan(hpr) else "N/A",
    "Time-Weighted Return (TWR)": round(twr, 4) if not np.isnan(twr) else "N/A",
    "Sharpe Ratio": round(sharpe_ratio, 4) if not np.isnan(sharpe_ratio) else "N/A",
    "Max Drawdown": round(max_drawdown, 4) if not np.isnan(max_drawdown) else "N/A",
    "Annualized Return": round(annualized_return, 4) if not np.isnan(annualized_return) else "N/A",
    "Annualized Volatility": round(annualized_volatility, 4) if not np.isnan(annualized_volatility) else "N/A",
    "Value at Risk (VaR 95%)": round(var_95, 4) if not np.isnan(var_95) else "N/A",
    "Win Rate (%)": round(win_rate * 100, 2) if not np.isnan(win_rate) else "N/A",
}

# Show trade log and performance metrics
# tools.display_dataframe_to_user(name="Optimized Trade Performance Log", dataframe=trade_df)
# tools.display_dataframe_to_user(name="Performance Metrics", dataframe=performance_metrics)

# Return calculated performance metrics
print(performance_metrics)
print(trade_df)
trade_df.to_csv('trade_df.csv', index=True)

In [2]:
import pandas as pd
import numpy as np
import glob
import os
import sys
import talib
import pandas_ta as pa_ta
import matplotlib.pyplot as plt

sys.path.append(os.path.dirname(os.path.realpath('__file__')) + '/../../')
from src.common.constants import *

def is_filtered_stock(symbol):
    filtered_list = [
        "CESC.NS",
        "DLF.NS",
        "ABFRL.NS",
        "LODHA.NS",
        "PVRINOX.NS",
        "POLYCAB.NS",
        "MGL.NS",
        "PEL.NS",
        "ABB.NS",
        "HUDCO.NS",
        "SUPREMEIND.NS",
        "ICICIPRULI.NS",
        "DIXON.NS",
        "SONACOMS.NS"
    ]
    
    return symbol in filtered_list

def is_nifty50_stock(symbol):
    nifty_50 = [
        "ADANIPORTS.NS",   # Adani Ports
        "APOLLOHOSP.NS",   # Apollo Hospitals
        "ASIANPAINT.NS",   # Asian Paints
        "AXISBANK.NS",     # Axis Bank
        "BAJAJ-AUTO.NS",   # Bajaj Auto
        "BAJFINANCE.NS",   # Bajaj Finance
        "BAJAJFINSV.NS",   # Bajaj Finserv
        "BPCL.NS",         # Bharat Petroleum
        "BHARTIARTL.NS",   # Bharti Airtel
        "BRITANNIA.NS",    # Britannia Industries
        "CIPLA.NS",        # Cipla
        "COALINDIA.NS",    # Coal India
        "DIVISLAB.NS",     # Divi's Laboratories
        "DRREDDY.NS",      # Dr. Reddy's
        "EICHERMOT.NS",    # Eicher Motors
        "GRASIM.NS",       # Grasim Industries
        "HCLTECH.NS",      # HCL Technologies
        "HDFCBANK.NS",     # HDFC Bank
        "HDFCLIFE.NS",     # HDFC Life Insurance
        "HEROMOTOCO.NS",   # Hero MotoCorp
        "HINDALCO.NS",     # Hindalco
        "HINDUNILVR.NS",   # Hindustan Unilever
        "ICICIBANK.NS",    # ICICI Bank
        "INDUSINDBK.NS",   # IndusInd Bank
        "INFY.NS",         # Infosys
        "ITC.NS",          # ITC Limited
        "JSWSTEEL.NS",     # JSW Steel
        "KOTAKBANK.NS",    # Kotak Mahindra Bank
        "LT.NS",           # Larsen & Toubro
        "M&M.NS",          # Mahindra & Mahindra
        "MARUTI.NS",       # Maruti Suzuki
        "NESTLEIND.NS",    # Nestle India
        "NTPC.NS",         # NTPC
        "ONGC.NS",         # Oil & Natural Gas Corp
        "POWERGRID.NS",    # Power Grid Corp
        "RELIANCE.NS",     # Reliance Industries
        "SBILIFE.NS",      # SBI Life Insurance
        "SBIN.NS",         # State Bank of India
        "SUNPHARMA.NS",    # Sun Pharma
        "TATAMOTORS.NS",   # Tata Motors
        "TATASTEEL.NS",    # Tata Steel
        "TCS.NS",          # Tata Consultancy Services
        "TECHM.NS",        # Tech Mahindra
        "TITAN.NS",        # Titan Company
        "ULTRACEMCO.NS",   # UltraTech Cement
        "UPL.NS",          # UPL Limited
        "WIPRO.NS",        # Wipro
        "ZOMATO.NS",       # Zomato (added recently)
        # Note: HDFC (merged with HDFC Bank in 2023) is excluded
    ]
    
    return symbol in nifty_50
    
    
def store_data(df, filename):
    # if filename exists then delete it and store the new one
    if os.path.exists(filename):
        os.remove(filename)
    df.to_csv(filename, index=True)

def compute_directional_indicators(df, window=14):
    # Ensure df has required columns
    high = df['High']
    low = df['Low']
    close = df['Close']

    # Calculate True Range (TR)
    df['prev_close'] = close.shift(1)
    df['high_low'] = high - low
    df['high_prev_close'] = (high - df['prev_close']).abs()
    df['low_prev_close'] = (low - df['prev_close']).abs()
    df['TR'] = df[['high_low', 'high_prev_close', 'low_prev_close']].max(axis=1)

    # Calculate Directional Movements (+DM and -DM)
    df['up_move'] = high.diff()  # current high - previous high
    df['up_move'] = np.where(df['up_move'] > 0, df['up_move'], 0)

    df['down_move'] = low.shift(1) - low  # previous low - current low
    df['down_move'] = np.where(df['down_move'] > 0, df['down_move'], 0)

    # Determine +DM and -DM
    df['+DM'] = np.where(df['up_move'] > df['down_move'], df['up_move'], 0)
    df['-DM'] = np.where(df['down_move'] > df['up_move'], df['down_move'], 0)

    # Apply Wilder's Smoothing (using exponential moving average)
    alpha = 1 / window  # Wilder's smoothing factor
    df['TR_sum'] = df['TR'].ewm(alpha=alpha, adjust=False).mean() * window
    df['+DM_sum'] = df['+DM'].ewm(alpha=alpha, adjust=False).mean() * window
    df['-DM_sum'] = df['-DM'].ewm(alpha=alpha, adjust=False).mean() * window

    # Calculate Directional Indicators
    df['Plus_DI'] = 100 * (df['+DM_sum'] / df['TR_sum'])
    df['Minus_DI'] = 100 * (df['-DM_sum'] / df['TR_sum'])

    # Clean up intermediate columns
    cols_to_drop = ['prev_close', 'high_low', 'high_prev_close', 'low_prev_close',
                    'up_move', 'down_move', '+DM', '-DM', 'TR_sum', '+DM_sum', '-DM_sum']
    df.drop(columns=cols_to_drop, inplace=True)

    return df

def compute_RSI(series, window=14):
    # Calculate price changes (delta)
    delta = series.diff()

    # Separate positive and negative changes
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # Calculate the Exponential Weighted Moving Average (Wilder's smoothing)
    avg_gain = gain.ewm(alpha=1/window, min_periods=window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/window, min_periods=window, adjust=False).mean()

    # Compute Relative Strength (RS)
    RS = avg_gain / avg_loss

    # Compute RSI using the formula
    RSI = 100 - (100 / (1 + RS))
    return RSI
   
def load_data(file_pattern):
    csv_files = glob.glob(file_pattern)
    dataframes = [pd.read_csv(file) for file in csv_files]
    merged_df = pd.concat(dataframes, ignore_index=True)
    merged_df['Datetime'] = pd.to_datetime(merged_df['Datetime'])
    merged_df = merged_df.sort_values(by='Datetime').reset_index(drop=True)
    merged_df.set_index('Datetime', inplace=True)
    return merged_df

# vma_window=20, roc_lookback=14, rolling_window=100, min_threshold=20, candle_lookback=5, atr_window=14
#TODO: Add price uptrend condition for buy signal i.e. last 5 candles High is uptrending 
#TODO: Add price downtrend condition for sell signal i.e. last 5 candles Low is downtrending
#TODO: Add OBR check 
def compute_indicators(df, vma_window, roc_lookback, rolling_window, min_threshold, candle_lookback, atr_window):
    df['VMA'] = df['Volume'].rolling(window=vma_window).mean()
    # df['VMA_ROC'] = ((df['VMA'] - df['VMA'].shift(roc_lookback)) / df['VMA'].shift(roc_lookback)) * 100
    # df['VMA_ROC_STD'] = df['VMA_ROC'].rolling(window=rolling_window).std()
    # df['Dynamic_Threshold'] = 1.5 * df['VMA_ROC_STD']
    # df['Final_Threshold'] = np.maximum(df['Dynamic_Threshold'], min_threshold)
    # df['VMA_ROC_Uptrend'] = df['VMA_ROC'].rolling(window=candle_lookback).apply(lambda x: all(x[i] > x[i-1] for i in range(1, len(x))), raw=True)
    df['ATR'] = talib.ATR(df['High'], df['Low'], df['Close'], timeperiod=atr_window)
    # df['ATR'] = ta.volatility.average_true_range(df['High'], df['Low'], df['Close'], window=atr_window)
    df['ADX_14'] = talib.ADX(df['High'], df['Low'], df['Close'], timeperiod=14)

    # df['ADX_21'] = ta.trend.adx(df['High'], df['Low'], df['Close'], window=21)
    # Add Volume uptrend condition which is looking at last 5 candles and all candle volume is twice the VMA
    # df['Volume_Uptrend'] = (df['Volume'] > 2 * df['VMA']).rolling(window=candle_lookback).apply(lambda x: x.all(), raw=True)
    
    # Ensure Volume_MA is shifted to use past data (avoiding lookahead bias)
    # Shift VMA to avoid lookahead bias
    df['Volume_MA_Shifted'] = df['VMA'].shift(1)

    # Condition: 70% of the last 7 candles should have volume > 1.2 * 30-period Volume MA
    def high_volume_condition(volume_window, volume_ma_shifted_window):
        if len(volume_window) < 7:
            return 0  # Not enough data points
        # Check if at least 4 of 7 volumes exceed 1.1x their respective shifted VMA
        return (volume_window > 1 * volume_ma_shifted_window).sum() >= 4

    df['Volume_Uptrend'] = df['Volume'].rolling(window=7).apply(
        lambda x: high_volume_condition(x, df['Volume_MA_Shifted'].loc[x.index]), raw=False
    )
    
    df['Stop_Loss'] = df['Close'] - (1.5 * df['ATR'])
    df['Trailing_Stop'] = df['Close'] - (1.5 * df['ATR'])
    # df['Price_High_Uptrend'] = df['High'].rolling(candle_lookback).apply(lambda x: x[-1] > x[:-1].max(), raw=True)
    # df['Price_Low_Downtrend'] = df['Low'].rolling(candle_lookback).apply(lambda x: x[-1] < x[:-1].min(), raw=True)
    # df_filtered = pd.concat([df.between_time('09:30', '11:30'), df.between_time('13:30', '15:00')])
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    compute_directional_indicators(df, window=14)
    df['RSI_14'] = compute_RSI(df['Close'], window=14)
    
    # Calculate Bollinger Bands
    # upper, middle, lower = talib.BBANDS(df['Close'], timeperiod=20,
    #         nbdevup=2, nbdevdn=2,  # standard deviations
    #         matype=talib.MA_Type.EMA  # exponential moving average
    # )
    # df[['BB_Upper', 'BB_Middle', 'BB_Lower']] = upper, middle, lower
    bbands = pa_ta.bbands(df['Close'], length=20, std=2, mamode='ema')
    df = pd.concat([df, bbands], axis=1)
    
    adx = pa_ta.adx(df['High'], df['Low'], df['Close'], length=14)
    df = pd.concat([df, adx], axis=1)
    return df

def generate_signals(df_filtered):
    # df_filtered['Buy_Signal'] = (df_filtered['VMA_ROC'] > df_filtered['Final_Threshold']) & (df_filtered['VMA_ROC_Uptrend'] == 1) & (df_filtered['Close'] > df_filtered['EMA_50'])
    # df_filtered['Sell_Signal'] = (df_filtered['VMA_ROC'] < -df_filtered['Final_Threshold']) & (df_filtered['Close'] < df_filtered['EMA_50'])
    # df_filtered.loc['2025-01-09'].to_csv('df_filtered.csv', index=True)
    # Buy condition if Volume_Uptrend and Price_High_Uptrend and ADX_14 > 25
    threshold = 0.005
    df_filtered['Buy_Signal'] = ((df_filtered['Close'] > df_filtered['EMA_50'] * (1 + threshold)) &
        (df_filtered['Volume_Uptrend'] == 1) &
        (df_filtered['ADX_14'] > 30) &
        (df_filtered['Plus_DI'] > df_filtered['Minus_DI'] * 1.1))
        
    df_filtered['Sell_Signal'] = (df_filtered['Close'] < df_filtered['EMA_50'] * (1 - threshold)) & (df_filtered['Plus_DI'] < df_filtered['Minus_DI'] * 0.9)
    return df_filtered

def plot_signals(df_filtered):
    plt.figure(figsize=(12, 6))
    plt.plot(df_filtered.index, df_filtered['VMA_ROC'], label="VMA-ROC", color='blue')
    plt.plot(df_filtered.index, df_filtered['Final_Threshold'], label="Buy Threshold", linestyle="dashed", color='green')
    plt.plot(df_filtered.index, -df_filtered['Final_Threshold'], label="Sell Threshold", linestyle="dashed", color='red')
    plt.scatter(df_filtered.index[df_filtered['Buy_Signal']], df_filtered['VMA_ROC'][df_filtered['Buy_Signal']], marker="^", color="green", label="Buy Signal", alpha=1)
    plt.scatter(df_filtered.index[df_filtered['Sell_Signal']], df_filtered['VMA_ROC'][df_filtered['Sell_Signal']], marker="v", color="red", label="Sell Signal", alpha=1)
    plt.title("VMA-ROC with Adaptive Threshold")
    plt.xlabel("Time")
    plt.ylabel("VMA-ROC (%)")
    plt.legend()
    plt.show()

def backtest_strategy(df_filtered, initial_capital):
    capital = initial_capital
    position = 0
    entry_price = 0
    highest_price = 0
    trade_log = []
    returns = []

    for i in range(len(df_filtered)):
        row = df_filtered.iloc[i]
        if row['Buy_Signal'] and position == 0 and row.name.time() <= pd.to_datetime('15:15').time()\
            and row.name.time() >= pd.to_datetime('14:30').time():
            position = capital // row['Close']
            entry_price = row['Close']
            highest_price = entry_price
            capital -= position * entry_price
            trade_log.append(('BUY', row.name, entry_price, position, capital, 0))
        if position > 0:
            highest_price = max(highest_price, row['Close'])
            trailing_stop_price = highest_price - (0.01 * entry_price)        
            stop_loss_price = entry_price - (0.01 * entry_price)
            
            if row['Sell_Signal'] or row['Close'] <= stop_loss_price or row['Close'] <= trailing_stop_price or row.name.time() >= pd.to_datetime('15:25').time():
                action = 'SELL' if row['Sell_Signal'] else 'STOP LOSS' if row['Close'] <= stop_loss_price else 'TRAILING STOP' if row['Close'] <= trailing_stop_price else 'TIME EXIT'
                sell_price = row['Close']
                capital += position * sell_price
                trade_return = (sell_price - entry_price) / entry_price
                returns.append(trade_return)
                trade_log.append((action, row.name, sell_price, position, capital, trade_return*100))
                position = 0
    if position > 0:
        sell_price = df_filtered['Close'].iloc[-1]
        capital += position * sell_price
        trade_return = (sell_price - entry_price) / entry_price
        returns.append(trade_return)
        trade_log.append(('FINAL EXIT', df_filtered.index[-1], sell_price, position, capital, trade_return*100))
        position = 0

    return trade_log, returns, capital

def calculate_performance_metrics(trade_log, returns, initial_capital, capital, df_filtered):
    net_profit = capital - initial_capital
    roi = (net_profit / initial_capital) * 100
    trade_df = pd.DataFrame(trade_log, columns=['Action', 'Datetime', 'Price', 'Shares', 'Capital', 'Return%'])
    hpr = np.mean(returns) if returns else np.nan
    twr = np.prod([1 + r for r in returns]) - 1 if returns else np.nan
    returns_series = pd.Series(returns) if returns else pd.Series([0])
    sharpe_ratio = returns_series.mean() / returns_series.std() * np.sqrt(252) if returns_series.std() > 0 else np.nan
    cumulative_returns = (1 + returns_series).cumprod() if returns else pd.Series([1])
    peak = cumulative_returns.cummax()
    drawdown = (cumulative_returns - peak) / peak
    max_drawdown = drawdown.min() if not drawdown.empty else np.nan
    trading_days = len(df_filtered) / 195
    annualized_return = ((1 + twr) ** (252 / trading_days)) - 1 if not np.isnan(twr) else np.nan
    annualized_volatility = returns_series.std() * np.sqrt(252) if returns_series.std() > 0 else np.nan
    var_95 = np.percentile(returns_series, 5) if not returns_series.empty else np.nan
    win_rate = sum(1 for r in returns if r > 0) / len(returns) if returns else 0

    performance_metrics = {
        "Net Profit (₹)": round(net_profit, 2),
        "ROI (%)": round(roi, 2),
        "Holding Period Return (HPR)": round(hpr, 4) if not np.isnan(hpr) else "N/A",
        "Time-Weighted Return (TWR)": round(twr, 4) if not np.isnan(twr) else "N/A",
        "Sharpe Ratio": round(sharpe_ratio, 4) if not np.isnan(sharpe_ratio) else "N/A",
        "Max Drawdown": round(max_drawdown, 4) if not np.isnan(max_drawdown) else "N/A",
        "Annualized Return": round(annualized_return, 4) if not np.isnan(annualized_return) else "N/A",
        "Annualized Volatility": round(annualized_volatility, 4) if not np.isnan(annualized_volatility) else "N/A",
        "Value at Risk (VaR 95%)": round(var_95, 4) if not np.isnan(var_95) else "N/A",
        "Win Rate (%)": round(win_rate * 100, 2) if not np.isnan(win_rate) else "N/A",
    }

    return performance_metrics, trade_df

def run_analysis(stock_symbol, initial_capital=100000, vma_window=20, roc_lookback=14, rolling_window=100, min_threshold=20, candle_lookback=5, atr_window=14):
    file_pattern = GlobalConstants.historicMarketData_dir + f'2m/{stock_symbol}*.csv'
    df = load_data(file_pattern)
    df_filtered = compute_indicators(df, vma_window, roc_lookback, rolling_window, min_threshold, candle_lookback, atr_window)
    df_filtered = generate_signals(df_filtered)
    store_data(df_filtered, f'/home/qa/runtime/data/analysis_out/{stock_symbol}.csv')
    # plot_signals(df_filtered)
    trade_log, returns, capital = backtest_strategy(df_filtered, initial_capital)
    performance_metrics, trade_df = calculate_performance_metrics(trade_log, returns, initial_capital, capital, df_filtered)
    return performance_metrics, trade_df

# Extract stock symbols from filenames
file_pattern = '/home/qa/runtime/data/historicMktData/2m/*.csv'
csv_files = glob.glob(file_pattern)
stock_symbols = list(set(os.path.basename(file).split('_')[0] for file in csv_files))
# stock_symbols = ['INFY.NS', 'ATGL.NS', 'CAMS.NS', 'EICHERMOT.NS', 'RELIANCE.NS', 'TCS.NS']
print(stock_symbols)
all_performance_df = None
all_trade_df = None
for stock_symbol in stock_symbols:
    if not is_nifty50_stock(stock_symbol):
        continue
    
    performance_metrics, trade_df = run_analysis(stock_symbol)
    print(f"Performance Metrics for {stock_symbol}:")
    print(performance_metrics)
    performance_df = pd.DataFrame([performance_metrics], index=[stock_symbol])
    # add stock_symbol in trade_df and index it by stock_symbol
    trade_df['Symbol'] = stock_symbol
    trade_df.set_index('Symbol', inplace=True)
    
    if all_performance_df is not None:
        all_performance_df = pd.concat([all_performance_df, performance_df])
    else:
        all_performance_df = performance_df
        
    if all_trade_df is not None:
        all_trade_df = pd.concat([all_trade_df, trade_df])
    else:
        all_trade_df = trade_df

# After the loop, save the concatenated dataframe to a CSV file
store_data(all_performance_df, '/home/qa/runtime/data/analysis_out/performance_metrics.csv')
store_data(all_trade_df, '/home/qa/runtime/data/analysis_out/trade_log.csv')
    

['LT.NS', 'ACC.NS', 'TECHM.NS', 'ONGC.NS', 'CANFINHOME.NS', 'CHAMBLFERT.NS', 'ABB.NS', 'YESBANK.NS', 'LTIM.NS', 'IOC.NS', 'HINDALCO.NS', 'IDFCFIRSTB.NS', 'MUTHOOTFIN.NS', 'GMRAIRPORT.NS', 'AARTIIND.NS', 'DEEPAKNTR.NS', 'KALYANKJIL.NS', 'ICICIBANK.NS', 'BEL.NS', 'KEI.NS', 'LALPATHLAB.NS', 'DRREDDY.NS', 'TATACONSUM.NS', 'BANDHANBNK.NS', 'SBICARD.NS', 'SHREECEM.NS', 'HDFCAMC.NS', 'CESC.NS', 'UBL.NS', 'BHARTIARTL.NS', 'RECLTD.NS', 'ALKEM.NS', 'AMBUJACEM.NS', 'TATACOMM.NS', 'TORNTPHARM.NS', 'APOLLOHOSP.NS', 'ESCORTS.NS', 'OBEROIRLTY.NS', 'IPCALAB.NS', 'OIL.NS', 'COROMANDEL.NS', 'PRESTIGE.NS', 'OFSS.NS', 'JIOFIN.NS', 'INFY.NS', 'IRFC.NS', 'KOTAKBANK.NS', 'TITAN.NS', 'SUPREMEIND.NS', 'TCS.NS', 'BAJFINANCE.NS', 'MAXHEALTH.NS', 'NATIONALUM.NS', 'PAGEIND.NS', 'LICI.NS', 'MANAPPURAM.NS', 'COALINDIA.NS', 'CANBK.NS', 'BATAINDIA.NS', 'ICICIGI.NS', 'BANKBARODA.NS', 'IGL.NS', 'GLENMARK.NS', 'ADANIENT.NS', 'PVRINOX.NS', 'INDIGO.NS', 'TATAMOTORS.NS', 'EICHERMOT.NS', 'METROPOLIS.NS', 'PAYTM.NS', 'APLAPOL

/tmp/ipykernel_1223606/2953167109.py:357: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_trade_df = pd.concat([all_trade_df, trade_df])


Performance Metrics for HINDUNILVR.NS:
{'Net Profit (₹)': -655.25, 'ROI (%)': -0.66, 'Holding Period Return (HPR)': -0.0022, 'Time-Weighted Return (TWR)': -0.0067, 'Sharpe Ratio': -7.1059, 'Max Drawdown': -0.0062, 'Annualized Return': -0.0373, 'Annualized Volatility': 0.0786, 'Value at Risk (VaR 95%)': -0.0059, 'Win Rate (%)': 33.33}
Performance Metrics for SUNPHARMA.NS:
{'Net Profit (₹)': -276.36, 'ROI (%)': -0.28, 'Holding Period Return (HPR)': -0.0006, 'Time-Weighted Return (TWR)': -0.0029, 'Sharpe Ratio': -2.6543, 'Max Drawdown': -0.0058, 'Annualized Return': -0.0161, 'Annualized Volatility': 0.0538, 'Value at Risk (VaR 95%)': -0.0048, 'Win Rate (%)': 40.0}
Performance Metrics for ITC.NS:
{'Net Profit (₹)': 372.95, 'ROI (%)': 0.37, 'Holding Period Return (HPR)': 0.0019, 'Time-Weighted Return (TWR)': 0.0037, 'Sharpe Ratio': 67.9747, 'Max Drawdown': 0.0, 'Annualized Return': 0.0215, 'Annualized Volatility': 0.0069, 'Value at Risk (VaR 95%)': 0.0016, 'Win Rate (%)': 100.0}
Performance

In [ ]:

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import gridspec
import pytz

def plot_trading_signals(df, timezone='Asia/Kolkata'):
    # Convert to local timezone (India)
    if df.index.tz is None:
        df = df.tz_localize('UTC').tz_convert(timezone)
    else:
        df = df.tz_convert(timezone)
    
    # Create figure with grid layout
    plt.figure(figsize=(16, 10))
    gs = gridspec.GridSpec(4, 1, height_ratios=[3, 1, 2, 1], hspace=0.3)
    
    # Custom color scheme
    price_color = '#2c3e50'
    ema_color = '#e74c3c'
    volume_color = '#3498db'
    vma_color = '#7f8c8d'
    threshold_green = '#27ae60'
    threshold_red = '#c0392b'

    # Price and EMA Plot (Top Panel)
    ax1 = plt.subplot(gs[0])
    ax1.plot(df.index, df['Close'], label='Close Price', color=price_color, lw=1.2)
    ax1.plot(df.index, df['EMA_50'], label='EMA 50', color=ema_color, lw=1, ls='--')
    
    # Plot buy/sell signals on price chart
    buy_signals = df[df['Buy_Signal']]
    sell_signals = df[df['Sell_Signal']]
    ax1.scatter(buy_signals.index, buy_signals['Close'], 
                marker='^', color=threshold_green, s=120, label='Buy Signal', zorder=5)
    ax1.scatter(sell_signals.index, sell_signals['Close'],
                marker='v', color=threshold_red, s=120, label='Sell Signal', zorder=5)
    
    ax1.set_title(f'Price Action with Signals ({df.index[0].strftime("%d %b %Y")})', fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper left')

    # Volume and VMA Plot (Second Panel)
    ax2 = plt.subplot(gs[1], sharex=ax1)
    ax2.bar(df.index, df['Volume'], color=volume_color, alpha=0.6, width=0.001)
    ax2.plot(df.index, df['VMA'], color=vma_color, lw=1, label='Volume MA')
    ax2.set_ylabel('Volume', fontsize=9)
    ax2.legend(loc='upper left')
    ax2.grid(True, alpha=0.3)

    # # VMA-ROC with Thresholds (Bottom Panel)
    # ax3 = plt.subplot(gs[3], sharex=ax1)
    # ax3.plot(df.index, df['VMA_ROC'], label='VMA ROC', color='#8e44ad', lw=1)
    # ax3.plot(df.index, df['Final_Threshold'], color=threshold_green, ls='--', lw=0.8)
    # ax3.plot(df.index, -df['Final_Threshold'], color=threshold_red, ls='--', lw=0.8)
    
    # # Fill between thresholds
    # ax3.fill_between(df.index, df['Final_Threshold'], -df['Final_Threshold'],
    #                 where=(df['VMA_ROC'] > df['Final_Threshold']),
    #                 facecolor=threshold_green, alpha=0.08)
    # ax3.fill_between(df.index, df['Final_Threshold'], -df['Final_Threshold'],
    #                 where=(df['VMA_ROC'] < -df['Final_Threshold']),
    #                 facecolor=threshold_red, alpha=0.08)
    
    # # Plot signals on ROC plot
    # ax3.scatter(buy_signals.index, buy_signals['VMA_ROC'],
    #            marker='^', color=threshold_green, s=80, zorder=5)
    # ax3.scatter(sell_signals.index, sell_signals['VMA_ROC'],
    #            marker='v', color=threshold_red, s=80, zorder=5)
    
    # ax3.axhline(0, color='black', lw=0.8, alpha=0.7)
    # ax3.set_ylabel('VMA ROC (%)', fontsize=9)
    # ax3.grid(True, alpha=0.3)

    # Time formatting for Indian market hours
    time_format = mdates.DateFormatter('%H:%M', tz=pytz.timezone(timezone))
    for ax in [ax1, ax2]:
        ax.xaxis.set_major_formatter(time_format)
        ax.xaxis.set_major_locator(mdates.HourLocator(interval=1))
        ax.xaxis.set_minor_locator(mdates.MinuteLocator(byminute=[15, 30, 45]))
        ax.tick_params(axis='x', which='major', labelsize=8, rotation=45)
        ax.tick_params(axis='y', which='major', labelsize=8)
        ax.set_xlim(df.index[0], df.index[-1])

    plt.suptitle(f'{stock_symbol} - 2min Chart Analysis (NSE India)', 
                y=0.95, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()



In [ ]:
import pandas as pd
stock_symbol = 'ZOMATO.NS'
df_plot = pd.read_csv(f'/home/qa/runtime/data/analysis_out/{stock_symbol}.csv')
df_plot['Datetime'] = pd.to_datetime(df_plot['Datetime'])
df_plot = df_plot.sort_values(by='Datetime').reset_index(drop=True)
df_plot.set_index('Datetime', inplace=True)
df_plot = df_plot.loc['2025-01-15']
# filter df_plot for 2024-12-23 and 2024-12-24
# df_plot = df_plot.loc['2024-12-23':'2024-12-24']

# Usage:
plot_trading_signals(df_plot)